In [3]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "name":   ["Aisha", "Bob", "Cy", "Dana", "Eli", "Farah"],
    "course": ["AI", "AI", "Data", "Data", "AI", "Data"],
    "quiz":   [85, 70, 90, 100, 55, 88],
    "final":  [88, 72, 91, 97, 60, 86],
})

In [3]:
print(df.groupby("course")["final"].mean())

course
AI      73.333333
Data    91.333333
Name: final, dtype: float64


In [4]:
print(df.groupby("course")[["quiz", "final"]].mean())

             quiz      final
course                      
AI      70.000000  73.333333
Data    92.666667  91.333333


In [5]:
print(df.groupby("course")["name"].count())

course
AI      3
Data    3
Name: name, dtype: int64


In [6]:
print(df.groupby("course")["final"].mean().sort_values(ascending=False))

course
Data    91.333333
AI      73.333333
Name: final, dtype: float64


In [8]:
summary = df.groupby("course").agg(students=("name", "count"), 
                     avg_final=("final", "mean"), best=("final", "max"))

print(summary)

        students  avg_final  best
course                           
AI             3  73.333333    88
Data           3  91.333333    97


In [9]:
print(summary.reset_index())

  course  students  avg_final  best
0     AI         3  73.333333    88
1   Data         3  91.333333    97


In [10]:
courses = pd.DataFrame({
    "course":   ["AI", "Data", "Cloud"],
    "lecturer": ["Dr. Lim", "Dr. Tan", "Dr. Wong"],
    "credits":  [4, 3, 2],
})

print(df.merge(courses, on="course", how="left"))

    name course  quiz  final lecturer  credits
0  Aisha     AI    85     88  Dr. Lim        4
1    Bob     AI    70     72  Dr. Lim        4
2     Cy   Data    90     91  Dr. Tan        3
3   Dana   Data   100     97  Dr. Tan        3
4    Eli     AI    55     60  Dr. Lim        4
5  Farah   Data    88     86  Dr. Tan        3


In [12]:
for how in ["left", "inner", "outer"]:
    print(how, df.merge(courses, on="course", how=how).shape)

left (6, 6)
inner (6, 6)
outer (7, 6)


In [4]:
m = pd.DataFrame({
    "name":  ["Aisha", "Bob", "Cy", "Dana", "Eli"],
    "quiz":  [85, np.nan, 90, 100, np.nan],
    "final": [88, 72, np.nan, 97, 60],
})

In [14]:
print(m)

    name   quiz  final
0  Aisha   85.0   88.0
1    Bob    NaN   72.0
2     Cy   90.0    NaN
3   Dana  100.0   97.0
4    Eli    NaN   60.0


In [15]:
print(m.isna())

    name   quiz  final
0  False  False  False
1  False   True  False
2  False  False   True
3  False  False  False
4  False   True  False


In [16]:
print(m.isna().sum())

name     0
quiz     2
final    1
dtype: int64


In [17]:
print(m.dtypes)

name         str
quiz     float64
final    float64
dtype: object


In [ ]:
print(m.dropna())
print(m.dropna(subset=["final"]))

    name   quiz  final
0  Aisha   85.0   88.0
1    Bob    NaN   72.0
3   Dana  100.0   97.0
4    Eli    NaN   60.0


In [20]:
m["quiz"] = m["quiz"].fillna(m["quiz"].mean())
print(m["quiz"].tolist())
print(m)

[85.0, 91.66666666666667, 90.0, 100.0, 91.66666666666667]
    name        quiz  final
0  Aisha   85.000000   88.0
1    Bob   91.666667   72.0
2     Cy   90.000000    NaN
3   Dana  100.000000   97.0
4    Eli   91.666667   60.0


In [21]:
dup = pd.DataFrame({
    "name":  ["Aisha", "Bob", "Aisha", "Cy", "Bob"],
    "score": [85, 70, 85, 90, 72],
})

In [25]:
print(dup.duplicated())
print(dup.duplicated().sum())
print(dup.drop_duplicates(subset="name"))

0    False
1    False
2     True
3    False
4    False
dtype: bool
1
    name  score
0  Aisha     85
1    Bob     70
3     Cy     90


In [29]:
new_dup = dup.drop_duplicates(subset="name", keep="last")
print(new_dup)
print(new_dup.reset_index(drop=True))

    name  score
2  Aisha     85
3     Cy     90
4    Bob     72
    name  score
0  Aisha     85
1     Cy     90
2    Bob     72


In [31]:
t = pd.DataFrame({
    "name":   ["  aisha ", "BOB", "cy"],
    "score":  ["85", "70", "abc"],
    "joined": ["2026-01-05", "2026-02-10", "not a date"],
})
print(t.dtypes)

name      str
score     str
joined    str
dtype: object


In [32]:
t["name"] = t["name"].str.strip().str.title()
t["score"] = pd.to_numeric(t["score"], errors="coerce")
t["joined"] = pd.to_datetime(t["joined"], errors="coerce")
print(t)
print(t.dtypes)

    name  score     joined
0  Aisha   85.0 2026-01-05
1    Bob   70.0 2026-02-10
2     Cy    NaN        NaT
name                 str
score            float64
joined    datetime64[us]
dtype: object


In [33]:
print(t["joined"].dt.month.tolist())     # [1.0, 2.0, nan]: .dt gives date parts
print(df["course"].unique())             # the distinct values in a column
print(df["course"].nunique())            # how many distinct values: 2

[1.0, 2.0, nan]
<StringArray>
['AI', 'Data']
Length: 2, dtype: str
2


In [1]:
import pandas as pd

raw = pd.read_csv("../../src/mini-projects/Day03/orders-cleaner/orders_raw.csv")

print(raw["quantity"].unique())      # which value is not a whole number?
print(raw["price"].unique())         # which value is not a number?
print(raw["price"].dtype)            # what type is it?

p = pd.to_numeric(raw["price"], errors="coerce")
print(p.dtype)                       # now a number column
print(p.isna().sum())                # 1 -> the "TBD" became NaN
print(raw.loc[p.isna(), ["order_id", "price"]])   # which order?

d = pd.to_datetime(raw["order_date"], errors="coerce")
print(d.dtype, d.isna().sum())       # 1 bad date
print(raw.loc[d.isna(), ["order_id", "order_date"]])   # which order?

[  1.   2.  nan  -1.   3. 500.]
<StringArray>
['3200', '45.5', '120', '850', 'TBD', '95']
Length: 6, dtype: str
str
float64
1
   order_id price
5      1006   TBD
datetime64[us] 1
   order_id  order_date
8      1008  2026-13-45


In [2]:
import pandas as pd

# ---- load (change the path if your notebook is in a different folder) ----
raw = pd.read_csv("../../src/mini-projects/Day03/orders-cleaner/orders_raw.csv")

# ---- the functions you already wrote ----
def clean_text(df):
    out = df.copy()
    for col in ["customer", "city", "product"]:
        out[col] = out[col].str.strip().str.title()
    return out

def convert_types(df):
    out = df.copy()
    out["quantity"] = pd.to_numeric(out["quantity"], errors="coerce")
    out["price"] = pd.to_numeric(out["price"], errors="coerce")
    out["order_date"] = pd.to_datetime(out["order_date"], errors="coerce")
    return out

def drop_duplicate_orders(df):
    return df.drop_duplicates()

def drop_missing(df):
    return df.dropna(subset=["quantity", "price", "order_date"])

def drop_bad_quantity(df):
    return df[df["quantity"] > 0]

def add_revenue(df):
    out = df.copy()
    out["revenue"] = out["quantity"] * out["price"]
    return out

# ---- rebuild the table after Piece 4 ----
step = clean_text(raw)
step = convert_types(step)
step = drop_duplicate_orders(step)
step = drop_missing(step)
step = drop_bad_quantity(step)
print("rows after Piece 4:", len(step))             # 8

# ---- Piece 5 experiment ----
s = add_revenue(step)
print(s[["order_id", "quantity", "price", "revenue"]])
print("sum  with outlier:", s["revenue"].sum())      # 32101.0
print("mean with outlier:", s["revenue"].mean())     # 4012.625
print("max revenue      :", s["revenue"].max())      # 22750.0

s2 = s[s["quantity"] <= 100]
print("rows without outlier:", len(s2))              # 7
print("sum  without outlier:", s2["revenue"].sum())  # 9351.0
print("mean without outlier:", s2["revenue"].mean()) # 1335.857...

# share of the total made by the biggest order, in percent
print("biggest order share:", round(s["revenue"].max() / s["revenue"].sum() * 100, 1))   # 70.9

rows after Piece 4: 8
    order_id  quantity   price  revenue
0       1001       1.0  3200.0   3200.0
1       1002       2.0    45.5     91.0
2       1003       1.0   120.0    120.0
3       1004       2.0   850.0   1700.0
9       1009       1.0   850.0    850.0
11      1010       1.0  3200.0   3200.0
12      1011     500.0    45.5  22750.0
13      1012       2.0    95.0    190.0
sum  with outlier: 32101.0
mean with outlier: 4012.625
max revenue      : 22750.0
rows without outlier: 7
sum  without outlier: 9351.0
mean without outlier: 1335.857142857143
biggest order share: 70.9


In [3]:
MAX_QUANTITY = 100

def remove_outliers(df, max_quantity=MAX_QUANTITY):
    return df[df["quantity"] <= max_quantity]

CATEGORIES = pd.DataFrame({
    "product": ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset", "Webcam"],
    "category": ["Computer", "Computer", "Accessory", "Accessory", "Accessory", "Accessory"],
})

clean = remove_outliers(add_revenue(step)).reset_index(drop=True)
print(len(clean))                                    # 7

print(clean.groupby("city")["revenue"].sum())        # a Series, city is the index
print(clean.groupby("city").agg(orders=("order_id", "count"),
                                revenue=("revenue", "sum")))
print(clean.groupby("city").agg(orders=("order_id", "count"),
                                revenue=("revenue", "sum")).reset_index())

with_cat = clean.merge(CATEGORIES, on="product", how="left")
print(with_cat.shape)                                # (7, 9)
print(with_cat["category"].isna().sum())             # 0
print(CATEGORIES.merge(clean, on="product", how="left").shape)   # (8, 9) - not 7. Why?

7
city
Johor Bahru      190.0
Kuala Lumpur    4141.0
Penang          5020.0
Name: revenue, dtype: float64
              orders  revenue
city                         
Johor Bahru        1    190.0
Kuala Lumpur       3   4141.0
Penang             3   5020.0
           city  orders  revenue
0   Johor Bahru       1    190.0
1  Kuala Lumpur       3   4141.0
2        Penang       3   5020.0
(7, 9)
0
(8, 9)
